# Exercise — Interpret Quality Results & Plan Remediation

Run **Trailhead Provisions**' quality suite (Great-Expectations-shaped results across six
dimensions). For each failure, choose a remediation strategy — **source**, **downstream**, or
**quarantine** — and write a prioritized plan tied to the platform's target contract SLOs. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")
q = gc.run_quality_suite()
print("Failing expectations:", int((~q["success"]).sum()), "of", len(q))
q

In [ ]:
display(q.groupby("dimension").agg(checks=("success", "size"),
                                   failing=("success", lambda s: int((~s).sum())),
                                   worst_pct=("unexpected_percent", "max")).reset_index())
print("--- Target Contract SLOs ---")
gc.contract_slos()

## 1. Classify each failing expectation
Map every `table.column` failure to `source` | `downstream` | `quarantine`.

In [ ]:
# TODO(you): map each failing table.column to "source" | "downstream" | "quarantine".
remediation = {
    # "product.category": "source",
}
fails = q[~q["success"]].assign(key=lambda d: d["table"] + "." + d["column"])
fails["strategy"] = fails["key"].map(remediation)
fails[["key", "dimension", "unexpected_percent", "strategy"]]

## 2. Prioritized remediation plan
Replace the cell below with your plan.

> **Your remediation plan here.** Address:
>
> - A strategy (`source` / `downstream` / `quarantine`) per issue, with justification.
> - Prioritization: which failures cause active GDPR or CSRD regulatory harm and must be quarantined first?
> - Tie every remediation tier to the target contract SLOs from `gc.contract_slos()`.
> - Which issues are recurring drift (consent, carbon unit) and need ongoing SLOs.